In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
import glob
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import torchvision.transforms as transforms
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def remap_mask(mask):
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

class SUIMDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, target_transform=None):
        self.image_paths = sorted(image_paths)
        self.mask_paths = sorted(mask_paths)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        if not isinstance(mask, torch.Tensor):
            mask = torch.tensor(np.array(mask))

        mask = remap_mask(mask)

        return image, mask


all_image_paths = sorted(glob.glob(os.path.join(path, "**", "images", "*.jpg"), recursive=True))
all_mask_paths = sorted(glob.glob(os.path.join(path, "**", "masks", "*.bmp"), recursive=True))

if not all_mask_paths:
     all_mask_paths = sorted(glob.glob(os.path.join(path, "**", "masks", "*.png"), recursive=True))


if len(all_image_paths) == 0:
    print("Warning: No images found. Check the dataset path structure.")


train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    all_image_paths, all_mask_paths, test_size=0.2, random_state=42
)


img_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


mask_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])


train_dataset = SUIMDataset(train_imgs, train_masks, transform=img_transform, target_transform=mask_transform)
val_dataset = SUIMDataset(val_imgs, val_masks, transform=img_transform, target_transform=mask_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"Training Samples: {len(train_dataset)}")
print(f"Validation Samples: {len(val_dataset)}")

# Display sample
img, mask = train_dataset[0]
print(f"Image Shape: {img.shape}, Mask Shape: {mask.shape}, Unique Classes: {torch.unique(mask)}")

In [ ]:
!pip install -q segmentation_models_pytorch
import segmentation_models_pytorch as smp
import torch

# TO DO
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8  # as you said
).to(device)

print("Model initialized and moved to device.")

In [ ]:
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim

# TO DO
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images = images.to(device)

        masks = masks.to(device).squeeze(1).long()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device).squeeze(1).long()

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)
num_epochs = 30

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import random

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.permute(1, 2, 0).cpu().numpy()
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

# TO DO
model.eval()
test_samples = random.sample(range(len(val_dataset)), 3)

for idx in test_samples:
    img, mask = val_dataset[idx]

    with torch.no_grad():
        input_tensor = img.unsqueeze(0).to(device)
        output = model(input_tensor)
        pred_mask = torch.argmax(output, dim=1).cpu().squeeze().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")


    axes[1].imshow(mask.squeeze(), cmap="jet", vmin=0, vmax=7)
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")


    axes[2].imshow(pred_mask, cmap="jet", vmin=0, vmax=7)
    axes[2].set_title("Prediction")
    axes[2].axis("off")

    plt.show()

In [ ]:
# again, not much but honest work